# Trading Strategy Test Lab

This notebook provides a lightweight sandbox for downloading market data, generating randomized strategies, running a simple backtest, and monitoring signals.

**Dependencies**: `pandas`, `numpy`, `matplotlib`, `yfinance`.
If needed: `pip install pandas numpy matplotlib yfinance`

In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import Callable, List, Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf


## Market Data
Download real price history from Yahoo Finance and compute indicators used by strategies.

In [ ]:
def download_market_data(ticker: str = "AAPL", period: str = "1y", interval: str = "1d") -> pd.DataFrame:
    data = yf.download(ticker, period=period, interval=interval, auto_adjust=True, progress=False)
    data = data.rename(columns=str.lower)
    data.index = pd.to_datetime(data.index)
    data = data.reset_index().rename(columns={"index": "date"})
    return data


def enrich_market_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["returns"] = df["close"].pct_change().fillna(0.0)

    for window in [5, 8, 10, 12, 20, 30, 40]:
        df[f"sma_{window}"] = df["close"].rolling(window).mean()

    for window in [3, 5, 7, 10]:
        df[f"momentum_{window}"] = df["close"].pct_change(window)

    for window in [10, 14, 20]:
        df[f"volatility_{window}"] = df["returns"].rolling(window).std()

    return df

ticker = "AAPL"
raw_data = download_market_data(ticker=ticker)
market_data = enrich_market_data(raw_data)
market_data.tail()

### Price & Indicator Plot

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(market_data["date"], market_data["close"], label="Close")
plt.plot(market_data["date"], market_data["sma_10"], label="SMA 10")
plt.plot(market_data["date"], market_data["sma_30"], label="SMA 30")
plt.title(f"{ticker} Close & Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

## Strategy Definitions
We randomly initialize a handful of strategies. Each strategy returns a target position: 
-1 = short, 0 = flat, 1 = long.

In [ ]:
SignalFunc = Callable[[pd.Series], int]

@dataclass
class Strategy:
    name: str
    signal: SignalFunc


def random_sma_strategy() -> Strategy:
    short = random.choice([5, 8, 10, 12])
    long = random.choice([20, 30, 40])
    name = f"SMA_{short}_{long}"

    def signal(row: pd.Series) -> int:
        if pd.isna(row[f"sma_{short}"]) or pd.isna(row[f"sma_{long}"]):
            return 0
        if row[f"sma_{short}"] > row[f"sma_{long}"]:
            return 1
        if row[f"sma_{short}"] < row[f"sma_{long}"]:
            return -1
        return 0

    return Strategy(name=name, signal=signal)


def random_momentum_strategy() -> Strategy:
    window = random.choice([3, 5, 7, 10])
    threshold = random.choice([0.01, 0.015, 0.02])
    name = f"MOM_{window}_{threshold:.3f}"

    def signal(row: pd.Series) -> int:
        momentum = row.get(f"momentum_{window}", np.nan)
        if pd.isna(momentum):
            return 0
        if momentum > threshold:
            return 1
        if momentum < -threshold:
            return -1
        return 0

    return Strategy(name=name, signal=signal)


def random_volatility_breakout() -> Strategy:
    vol_window = random.choice([10, 14, 20])
    breakout = random.choice([0.02, 0.025, 0.03])
    name = f"VOL_{vol_window}_{breakout:.3f}"

    def signal(row: pd.Series) -> int:
        vol = row.get(f"volatility_{vol_window}", np.nan)
        if pd.isna(vol):
            return 0
        if row["returns"] > breakout + vol:
            return 1
        if row["returns"] < -(breakout + vol):
            return -1
        return 0

    return Strategy(name=name, signal=signal)


def bootstrap_strategies(count: int = 5) -> List[Strategy]:
    strategy_builders = [
        random_sma_strategy,
        random_momentum_strategy,
        random_volatility_breakout,
    ]
    return [random.choice(strategy_builders)() for _ in range(count)]


strategies = bootstrap_strategies()
[strategy.name for strategy in strategies]

## Backtesting Engine
Runs a simple daily rebalancing backtest for each strategy.

In [ ]:
@dataclass
class BacktestResult:
    strategy: Strategy
    equity_curve: pd.Series
    stats: Dict[str, float]


def run_backtest(data: pd.DataFrame, strategy: Strategy, initial_cash: float = 10000.0) -> BacktestResult:
    positions = data.apply(strategy.signal, axis=1).shift(1).fillna(0)
    strategy_returns = positions * data["returns"]
    equity_curve = (1 + strategy_returns).cumprod() * initial_cash

    total_return = equity_curve.iloc[-1] / initial_cash - 1
    annualized_return = (1 + total_return) ** (252 / len(data)) - 1
    volatility = strategy_returns.std() * math.sqrt(252)
    sharpe = (strategy_returns.mean() * 252) / volatility if volatility else 0.0
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()

    stats = {
        "total_return": total_return,
        "annualized_return": annualized_return,
        "volatility": volatility,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
    }
    return BacktestResult(strategy=strategy, equity_curve=equity_curve, stats=stats)


results = [run_backtest(market_data, strategy) for strategy in strategies]
pd.DataFrame(
    [{"strategy": r.strategy.name, **r.stats} for r in results]
).sort_values(by="sharpe", ascending=False)

### Equity Curve Comparison

In [ ]:
plt.figure(figsize=(12, 5))
for result in results:
    plt.plot(market_data["date"], result.equity_curve, label=result.strategy.name)
plt.title("Strategy Equity Curves")
plt.xlabel("Date")
plt.ylabel("Equity")
plt.legend()
plt.tight_layout()
plt.show()

## Monitoring Dashboard
Capture the latest signal and market context for each strategy.

In [ ]:
def build_monitoring_snapshot(data: pd.DataFrame, strategies: List[Strategy]) -> pd.DataFrame:
    latest = data.iloc[-1]
    snapshot = []
    for strategy in strategies:
        signal = strategy.signal(latest)
        snapshot.append({
            "strategy": strategy.name,
            "signal": signal,
            "close": latest["close"],
            "momentum_7": latest.get("momentum_7", np.nan),
            "volatility_14": latest.get("volatility_14", np.nan),
        })
    return pd.DataFrame(snapshot)

monitoring_snapshot = build_monitoring_snapshot(market_data, strategies)
monitoring_snapshot

## Next Steps
- Replace the single ticker with a basket of assets.
- Add transaction costs and position sizing.
- Use the monitoring snapshot to trigger alerts or a dashboard.